<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_04_regression_and_prediction/04_regression_and_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 4 folder
%cd chapter_04_regression_and_prediction

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (133/133), done.
Receiving objects: 100% (172/172), 586.37 KiB | 17.25 MiB/s, done.
remote: Total 172 (delta 105), reused 70 (delta 37), pack-reused 0 (from 0)
Resolving deltas: 100% (105/105), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_04_regression_and_prediction


# Chapter 04: Regression and Prediction

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 4: Regression and Prediction

## Goals

In this notebook, I will learn:
- Simple linear regression
- Multiple linear regression
- Interpreting coefficients
- Model evaluation (RMSE, R², Cross-Validation)
- Handling categorical (factor) variables
- Dummy encoding and high-cardinality features
- Interaction effects
- Confounding variables
- Regression diagnostics (residuals, outliers, influential values, heteroskedasticity)
- Polynomial and spline regression
- Generalized Additive Models (GAMs)

---

## 🎯 Learning Objectives

1. Fit and interpret simple and multiple linear regression models
2. Evaluate model performance using RMSE, R², and Cross-Validation
3. Handle categorical (factor) variables using dummy encoding and manage high-cardinality features
4. Interpret coefficients, identify confounding variables, and model interactions
5. Diagnose regression problems: outliers, influential values, and heteroskedasticity
6. Extend linear models to capture nonlinearity using Polynomials, Splines, and GAMs

---

## 🤖 ML Relevance

> Why does regression matter for Machine Learning?
> * **Baseline Models**: Linear regression is the ultimate baseline. If a complex neural network can't beat OLS, your complex model is overfitting or poorly tuned.
> * **Interpretability**: In regulated industries (finance, healthcare), linear models are often required because their coefficients provide transparent, explainable decisions.
> * **Feature Engineering**: Understanding interactions and nonlinearities (splines) helps you create better features for tree-based models and neural networks.
> * **Diagnostics**: Residual analysis is crucial for detecting data leakage, concept drift, and heteroskedasticity in production models.

---

## Setup & Data Preparation

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    from scipy import stats
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, KFold

# Helper to load book datasets
def load_data(filename, url):
    local_path = f'../datasets/raw/{filename}'
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    else:
        print(f"Downloading {filename}...")
        return pd.read_csv(url)

# 1. Lung Data (Simple Linear Regression)
lung_url = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/lung.csv"
lung = load_data('lung.csv', lung_url)

# 2. King County Housing Data (Multiple Regression)
house_url = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/kc_house_data.csv"
house = load_data('kc_house_data.csv', house_url)

# Safeguard: Map 'price' to 'AdjSalePrice' if the column name differs
if 'AdjSalePrice' not in house.columns and 'price' in house.columns:
    house['AdjSalePrice'] = house['price']
    
# Create ZipGroup for later confounding analysis
try:
    house['ZipGroup'] = pd.qcut(house['ZipCode'].value_counts().index, q=5, labels=False, duplicates='drop')
except:
    house['ZipGroup'] = pd.cut(house['ZipCode'], bins=5, labels=False)

print("Datasets loaded successfully.")
print(f"Lung data shape: {lung.shape}")
print(f"House data shape: {house.shape}")
```
</details>


---

# Section 1: Simple Linear Regression

## What is Simple Linear Regression?

Simple linear regression models the relationship between a single predictor X and a continuous response Y as a straight line:

$$Y = \beta_0 + \beta_1X + \epsilon$$

- **β₀ (Intercept)**: The predicted value of Y when X = 0
- **β₁ (Slope/Coefficient)**: The expected change in Y for a 1-unit increase in X
- **ε (Error/Residual)**: The difference between the observed Y and the predicted Ŷ

The model is fit using **Ordinary Least Squares (OLS)**, which minimizes the Residual Sum of Squares (RSS).

<details>
<summary>Click to view solution</summary>

```python
# Simple Linear Regression: PEFR vs Exposure
model_simple = smf.ols('PEFR ~ Exposure', data=lung).fit()
print(model_simple.summary())

# Visualizing the fit
plt.figure(figsize=(8, 5))
sns.scatterplot(data=lung, x='Exposure', y='PEFR', alpha=0.6)
plt.plot(lung['Exposure'], model_simple.fittedvalues, color='red', linewidth=2, label='OLS Fit')
plt.xlabel('Years of Exposure to Cotton Dust')
plt.ylabel('PEFR (Lung Capacity)')
plt.title('Simple Linear Regression: PEFR vs Exposure')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Interpretation: For every additional year of exposure, PEFR decreases by {abs(model_simple.params['Exposure']):.2f} units.")
```
</details>


---

# Section 2: Simple Regression with Simulated Data

## Visual Intuition

<details>
<summary>Click to view solution</summary>

```python
# Generate simple regression data
np.random.seed(42)
X_simple = np.random.normal(50, 15, 200)
y_simple = X_simple * 3 + np.random.normal(0, 20, 200)

plt.figure(figsize=(8, 6))
plt.scatter(X_simple, y_simple, alpha=0.5)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Simple Linear Relationship")
plt.grid(alpha=0.3)
plt.show()
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Fit simple linear regression
simple_model = LinearRegression()
simple_model.fit(X_simple.reshape(-1, 1), y_simple)

print(f"Intercept: {simple_model.intercept_:.2f}")
print(f"Coefficient: {simple_model.coef_[0]:.2f}")

# Predictions
y_pred_simple = simple_model.predict(X_simple.reshape(-1, 1))

plt.figure(figsize=(8, 6))
plt.scatter(X_simple, y_simple, alpha=0.5, label="Actual")
plt.plot(X_simple, y_pred_simple, color='red', linewidth=2, label="Regression Line")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Simple Linear Regression Fit")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"R² Score: {r2_score(y_simple, y_pred_simple):.4f}")
```
</details>

## Reflection

Questions:
1. What does the coefficient mean?
2. Is the relationship strong?
3. Why do points scatter around the line?



---

# Section 3: Model Evaluation

## Key Metrics

- **RMSE (Root Mean Squared Error)**: The standard deviation of the residuals. Tells you how far off predictions are in the units of Y.
- **R² (Coefficient of Determination)**: The proportion of variance in Y explained by the model (0 to 1).
- **Adjusted R²**: Penalizes R² for adding useless predictors.

<details>
<summary>Click to view solution</summary>

```python
rmse_simple = np.sqrt(mean_squared_error(y_simple, y_pred_simple))
r2_simple = r2_score(y_simple, y_pred_simple)

print(f"RMSE: {rmse_simple:.2f}")
print(f"R²: {r2_simple:.4f}")
print(f"RMSE represents the typical prediction error in units of Y.")
print(f"R² of {r2_simple:.4f} means {r2_simple*100:.1f}% of variance is explained.")
```
</details>

## Reflection

Questions:
1. Is R² of 0.75 good?
2. What does RMSE tell us?
3. Why do we need both metrics?


---

# Section 4: Multiple Linear Regression

## What is multiple regression?

When we have multiple predictors, the equation extends linearly:

$$Y = \beta_0 + \beta_1X_1 + \beta_2X_2 + \dots + \beta_pX_p + \epsilon$$

<details>
<summary>Click to view solution</summary>

```python
# Simulated housing data
np.random.seed(42)

housing_data = pd.DataFrame({
    "House_Size": np.random.normal(1500, 300, 300),
    "Bedrooms": np.random.randint(1, 6, 300),
    "House_Age": np.random.randint(1, 50, 300)
})

housing_data["House_Price"] = (
    housing_data["House_Size"] * 200 +
    housing_data["Bedrooms"] * 10000 -
    housing_data["House_Age"] * 800 +
    np.random.normal(0, 30000, 300)
)

# Features and target
X_multi = housing_data[["House_Size", "Bedrooms", "House_Age"]]
y_multi = housing_data["House_Price"]

# Fit model
multiple_model = LinearRegression()
multiple_model.fit(X_multi, y_multi)

coefficients = pd.DataFrame({
    "Feature": X_multi.columns,
    "Coefficient": multiple_model.coef_
})
print(coefficients)
print(f"\nIntercept: {multiple_model.intercept_:.2f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Using King County Housing Data
predictors = ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade']
outcome = 'AdjSalePrice'

X_house = house[predictors].dropna()
y_house = house.loc[X_house.index, outcome]

sk_model = LinearRegression()
sk_model.fit(X_house, y_house)

y_pred_house = sk_model.predict(X_house)
rmse_house = np.sqrt(mean_squared_error(y_house, y_pred_house))
r2_house = r2_score(y_house, y_pred_house)

print(f"RMSE: ${rmse_house:,.0f}")
print(f"R²: {r2_house:.4f}")

print("\nCoefficients:")
for name, coef in zip(predictors, sk_model.coef_):
    print(f"  {name}: {coef:,.2f}")
```
</details>

## Reflection

Questions:
1. Which feature matters most?
2. Why is house age negative?
3. Why are coefficients useful?



---

# Section 5: Interpreting Coefficients

Each coefficient means: Expected change in Y for one unit increase in X, while holding everything else constant.

⚠️ Important: Regression finds association, not necessarily causation.

<details>
<summary>Click to view solution</summary>

```python
predictions_multi = multiple_model.predict(X_multi)
r2_multi = r2_score(y_multi, predictions_multi)

print(f"R² Score: {r2_multi:.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_multi, predictions_multi)):.2f}")

print("\nInterpretation:")
print(f"- Each additional sq ft adds ~${multiple_model.coef_[0]:.0f}")
print(f"- Each additional bedroom adds ~${multiple_model.coef_[1]:.0f}")
print(f"- Each year of age reduces price by ~${abs(multiple_model.coef_[2]):.0f}")
```
</details>

## Reflection

Questions:
1. Why might high R² still be misleading?
2. Does prediction mean causation?
3. Can important variables be missing?



---

# Section 6: Cross-Validation

Classic metrics like R² and RMSE calculated on the training data are in-sample metrics. They are overly optimistic because the model was optimized for that specific data.

**K-Fold Cross-Validation** splits the data into K folds, trains on K-1 folds, and tests on the held-out fold. This provides a realistic out-of-sample estimate of predictive performance.

<details>
<summary>Click to view solution</summary>

```python
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(sk_model, X_house, y_house, cv=kf, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)

print(f"CV RMSE scores: {cv_rmse}")
print(f"Mean CV RMSE: ${cv_rmse.mean():,.0f} ± ${cv_rmse.std():,.0f}")
print(f"\nIn-sample RMSE: ${rmse_house:,.0f}")
print(f"CV RMSE: ${cv_rmse.mean():,.0f}")
print("The gap between in-sample and CV RMSE represents overfitting.")
```
</details>


---

# Section 7: Categorical Variables & Dummy Encoding

## Problem

Regression expects numbers, but real-world data contains categories (cabin type, airline, customer segment, region).

## Solution

Convert categories into 0s and 1s using dummy encoding.

### Why drop_first=True?

Avoids the dummy variable trap which causes perfect multicollinearity.

<details>
<summary>Click to view solution</summary>

```python
# Create example airline data
airline_df = pd.DataFrame({
    "Cabin": np.random.choice(["economy", "business", "first"], size=200),
    "Spend": np.random.normal(300, 50, 200)
})

print("Original data:")
print(airline_df.head())

# Dummy encoding
encoded_df = pd.get_dummies(airline_df, columns=["Cabin"], drop_first=True)
print("\nEncoded data:")
print(encoded_df.head())

# Fit model with categorical variables
X_cat = encoded_df.drop("Spend", axis=1)
y_cat = encoded_df["Spend"]

cabin_model = LinearRegression()
cabin_model.fit(X_cat, y_cat)

coef_df = pd.DataFrame({
    "Feature": X_cat.columns,
    "Coefficient": cabin_model.coef_
})
print("\nCoefficients:")
print(coef_df)
```
</details>

## Reflection

Questions:
1. What does positive coefficient mean?
2. Why must categories become numbers?
3. Why remove one category?


---

# Section 8: Factor Variables with Housing Data

<details>
<summary>Click to view solution</summary>

```python
# Statsmodels handles categorical variables using the C() operator
formula = 'AdjSalePrice ~ SqFtTotLiving + Bathrooms + Bedrooms + BldgGrade + C(PropertyType) + C(ZipGroup)'
model_factors = smf.ols(formula, data=house.dropna(subset=['PropertyType', 'ZipGroup'])).fit()

# Extract categorical coefficients
cat_coefs = model_factors.params[model_factors.params.index.str.contains('PropertyType|ZipGroup')]
print("Categorical Coefficients (Relative to Baseline):")
print(cat_coefs.round(2))
```
</details>


---

# Section 9: Interaction Effects

## Key idea

Sometimes one variable changes the effect of another.

Example: Study hours may matter differently for beginners vs experts.

<details>
<summary>Click to view solution</summary>

```python
# Create student data with interaction
student_df = pd.DataFrame({
    "Study_Hours": np.random.randint(1, 10, 300),
    "Experience": np.random.randint(0, 2, 300)
})

student_df["Score"] = (
    student_df["Study_Hours"] * 5 +
    student_df["Experience"] * 10 +
    (student_df["Study_Hours"] * student_df["Experience"] * 2) +
    np.random.normal(0, 5, 300)
)

# Create interaction term
student_df["Interaction"] = student_df["Study_Hours"] * student_df["Experience"]

X_int = student_df[["Study_Hours", "Experience", "Interaction"]]
y_int = student_df["Score"]

interaction_model = LinearRegression()
interaction_model.fit(X_int, y_int)

coef_int = pd.DataFrame({
    "Feature": X_int.columns,
    "Coefficient": interaction_model.coef_
})
print(coef_int)
print(f"Intercept: {interaction_model.intercept_:.2f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Housing data interaction: Location × Size
interaction_formula = 'AdjSalePrice ~ SqFtTotLiving * C(ZipGroup) + Bathrooms + BldgGrade'
model_interact = smf.ols(interaction_formula, data=house.dropna(subset=['ZipGroup'])).fit()

interact_terms = model_interact.params[model_interact.params.index.str.contains(':')]
print("Interaction Terms (SqFtTotLiving : ZipGroup):")
print(interact_terms.round(2))
print("\nInsight: The premium for square footage scales up dramatically in higher ZipGroups.")
```
</details>

## Reflection

Questions:
1. Did experience change study effect?
2. Why do interactions matter?
3. Can business behaviour depend on context?


---

# Section 10: Confounding Variables

## What is confounding?

Sometimes hidden variables create misleading relationships.

Example: Ice cream sales and drowning are correlated, but summer weather causes both.

⚠️ Regression can show correlation, but not automatically causation.

<details>
<summary>Click to view solution</summary>

```python
# Generate confounding example
temperature = np.random.normal(30, 5, 200)
ice_cream_sales = temperature * 10 + np.random.normal(0, 5, 200)
swimming = temperature * 8 + np.random.normal(0, 5, 200)

confounding_df = pd.DataFrame({
    "Temperature": temperature,
    "Ice_Cream": ice_cream_sales,
    "Swimming": swimming
})

print("Correlation matrix:")
print(confounding_df.corr().round(3))
print("\nIce cream and swimming are correlated, but temperature is the true cause.")
```
</details>

## Reflection

Questions:
1. Why are variables correlated?
2. Does ice cream cause swimming?
3. Why are confounders dangerous?


---

# Section 11: Regression Diagnostics

Analyzing residuals (eᵢ = Yᵢ - Ŷᵢ) helps us find where the model fails.

1. **Outliers**: Records with massive standardized residuals (> 3)
2. **Influential Values (Cook's Distance)**: Records that would drastically change coefficients if removed
3. **Heteroskedasticity**: When variance of residuals changes with predicted value (funnel shape)

<details>
<summary>Click to view solution</summary>

```python
# Residual Analysis & Cook's Distance
diag_model = smf.ols('AdjSalePrice ~ SqFtTotLiving + BldgGrade',
                      data=house.sample(2000, random_state=42)).fit()

influence = diag_model.get_influence()
cooks_d = influence.cooks_distance[0]
std_resid = influence.resid_studentized_internal

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Heteroskedasticity Check (Residuals vs Fitted)
sns.scatterplot(x=diag_model.fittedvalues, y=std_resid, alpha=0.5, ax=axes[0])
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Standardized Residuals')
axes[0].set_title('Residuals vs Fitted (Look for funnel shapes)')

# 2. Influential Points (Cook's Distance)
axes[1].stem(np.arange(len(cooks_d)), cooks_d, markerfmt=",", linefmt='C0-')
threshold = 4 / len(cooks_d)
axes[1].axhline(threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.3f})')
axes[1].set_xlabel('Observation Index')
axes[1].set_ylabel("Cook's Distance")
axes[1].set_title("Influential Observations")
axes[1].legend()

plt.tight_layout()
plt.show()
```
</details>


---

# Section 12: Residual Analysis

<details>
<summary>Click to view solution</summary>

```python
# Residual plot for simple model
residuals = y_simple - y_pred_simple

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_simple, residuals, alpha=0.5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel("Predicted Values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residual Plot")

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel("Residuals")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()
```
</details>

## Reflection

Questions:
1. Are residuals random?
2. Do you see patterns?
3. Why should residuals be random?


---

# Section 13: Overfitting Intuition

<details>
<summary>Click to view solution</summary>

```python
# Overfitting demonstration
np.random.seed(42)
X_overfit = np.random.normal(50, 15, 50).reshape(-1, 1)
y_overfit = X_overfit.ravel() * 3 + np.random.normal(0, 15, 50)

# Train and test split
train_size = 35
X_train = X_overfit[:train_size]
y_train = y_overfit[:train_size]
X_test = X_overfit[train_size:]
y_test = y_overfit[train_size:]

# Simple model
simple_overfit = LinearRegression()
simple_overfit.fit(X_train, y_train)

train_score = simple_overfit.score(X_train, y_train)
test_score = simple_overfit.score(X_test, y_test)

print(f"Training R²: {train_score:.4f}")
print(f"Test R²: {test_score:.4f}")
print(f"Gap (overfitting indicator): {train_score - test_score:.4f}")
```
</details>

## Reflection

Questions:
1. Why is training score higher?
2. Why does overfitting matter?
3. How does CV help?


---

# Section 14: Nonlinear Relationships

Real-world relationships are rarely perfectly linear.

- **Polynomial Regression**: Adds squared or cubed terms (X², X³). Simple but can behave wildly at edges.
- **Splines**: Piecewise polynomials joined smoothly at "knots". Much more stable than high-degree polynomials.
- **Generalized Additive Models (GAMs)**: Automatically fits splines to predictors.

<details>
<summary>Click to view solution</summary>

```python
# GAM Example
try:
    from pygam import LinearGAM, s, l
    
    X_gam = house[['SqFtTotLiving', 'Bathrooms', 'BldgGrade']].dropna()
    y_gam = house.loc[X_gam.index, 'AdjSalePrice']
    
    gam = LinearGAM(s(0, n_splines=10) + l(1) + l(2)).fit(X_gam.values, y_gam.values)
    
    XX = gam.generate_X_grid(term=0)
    plt.figure(figsize=(8, 5))
    plt.plot(XX[:, 0], gam.partial_dependence(term=0, X=XX), color='blue', linewidth=2)
    plt.fill_between(XX[:, 0],
                     gam.partial_dependence(term=0, X=XX, width=-1)[0],
                     gam.partial_dependence(term=0, X=XX, width=-1)[1],
                     alpha=0.2, color='blue')
    plt.title('GAM: Non-linear Effect of SqFtTotLiving on Price')
    plt.xlabel('SqFtTotLiving')
    plt.ylabel('Partial Dependence (Price Impact)')
    plt.grid(alpha=0.3)
    plt.show()
    
except ImportError:
    print("pygam is not installed. Run `pip install pygam` to execute GAM models.")
```
</details>


---

# Mini Experiment

## Question

Remove one predictor (e.g., Bedrooms) and observe how coefficients change.

Questions:
1. Did coefficients shift?
2. Why?
3. What does this teach about missing variables?

---

# ML Relevance

## Why Chapter 4 matters in Machine Learning

- Regression is the foundation of prediction
- Linear models are often the best baseline
- Understanding coefficients builds intuition
- Residual analysis reveals data problems
- Overfitting is a constant danger
- Cross-validation is essential
- Feature engineering matters

### ML Relevance Recap

| Concept | ML Application |
|---------|---------------|
| **Cross-Validation** | The gold standard for hyperparameter tuning and preventing overfitting |
| **Regularization (Ridge/Lasso)** | Penalized regression is the foundation of modern linear classifiers |
| **Partial Dependence** | GAM partial dependence plots are identical to PDPs used to interpret RFs and XGBoost |
| **Residual Analysis** | Used in production to monitor data drift |

---

# Interview Questions

1. **What is linear regression?**
2. **What does R² tell us?**
3. **Difference between simple and multiple regression?**
4. **What are dummy variables?**
5. **Why avoid dummy variable trap?**
6. **What is interaction effect?**
7. **What is confounding?**
8. **Why do we need cross-validation?**
9. **What is RMSE?**
10. **Why check residuals?**
11. **What is heteroskedasticity?**
12. **Why might high R² be misleading?**

---

# Chapter Summary

## Key Concepts Learned
- Simple and multiple linear regression
- Model evaluation (RMSE, R², Cross-Validation)
- Categorical variable encoding
- Interaction effects and confounding
- Regression diagnostics (residuals, outliers, Cook's Distance)
- Nonlinear extensions (Polynomials, Splines, GAMs)

## Biggest takeaway

Regression is about understanding relationships and making predictions. Coefficients show association, not necessarily causation. Always validate with cross-validation and check residuals.

---

# Progress Checklist

- [ ] Understood simple linear regression
- [ ] Built multiple regression models
- [ ] Interpreted coefficients
- [ ] Evaluated models with RMSE and R²
- [ ] Applied cross-validation
- [ ] Handled categorical variables with dummy encoding
- [ ] Modeled interaction effects
- [ ] Identified confounding variables
- [ ] Analysed residuals for diagnostics
- [ ] Checked for heteroskedasticity
- [ ] Identified influential observations
- [ ] Explored nonlinear relationships (GAMs)
- [ ] Connected concepts to ML
- [ ] Ready for Chapter 5

---

> **Pro Tip**: Don't just look at the R². Spend time analyzing the residual plots. In the real world, diagnosing *why* a model is failing is more valuable than just fitting it.